# 04: 多方法嵌入比较——PI 工作界面

运行 PCA（基线）、Harmony、scVI（从头训练），以及可选的
scANVI（存在标注参考时）和 cellxgene_census 预训练 scVI。

所有嵌入方法**等权并列**——无优先级预设。选择依据：
1. **可视化检查（首要）**——每个嵌入算完**即时出 UMAP 图**，按样本、批次、
   细胞类型三着色。PI 跑一个看一个，目视判断批次整合与生物学信号保留的平衡。
2. **整合指标（佐证）**——全部嵌入跑完后，显式 for 循环调
   `integration_metrics(adata)` 生成对比表。无回调、无 sweep() 抽象。

**本 notebook 产出**：
- `obsm["X_pca"]`——PCA（基线，仅 HVG）
- `obsm["X_pca_harmony"]`——Harmony 批次校正 PCA
- `obsm["X_scVI"]`——scVI 潜变量（从头训练）
- `obsm["X_scANVI"]`——scANVI 潜变量（仅当存在标注参考时）
- 每个嵌入的 UMAP 图：计算后立即出图（三着色），跑一个看一个
- `results/figures/sweep_04/` 中显式遍历产出对比表（含整合指标）
- `adata.uns` 运行元数据（`harmony_v1`、`scvi_v1` 等）
- 04 checkpoint h5ad，供 05 聚类使用

**增加新嵌入方法**（扩展模式）：写 `adata.obsm["X_{method}"]`，
并将 `"X_{method}"` 加入整合指标 cell 的 `use_reps` 列表——一个 cell，零框架改动。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：03（标准化 + HVG），读 `03_normalized_v*.h5ad`
- **下游**：05（多分辨率 Leiden 聚类），产出 `04_embedded_v*.h5ad`

### 为什么要迭代回跑？
嵌入质量直接影响下游分群和注释的准确性。如果在 05（Leiden 分群不合理）、
06（注释时发现嵌入没有分离应区分的细胞类型）发现问题，可能需要：
- 换用不同的嵌入方法（增减 `EMBEDDING_METHODS` 列表）
- 调整 PCA 维度数（`N_PCS`）
- 增加 scVI 训练轮数（`SCVI_MAX_EPOCHS`——默认 20 是快速验证用，生产建议 200-400）
- 换用 03 的另一个版本（不同 HVG 数量）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `03_normalized_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `04_embedded_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改嵌入方法或训练参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为嵌入质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：05 的 `UPSTREAM_PATH` 指向你决定采用的 04 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"04_embedded"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"04 有哪些版本？哪些依赖 03_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===

UPSTREAM_PATH = "results/03_normalized_v1.h5ad"
OUTPUT_PATH   = "results/04_embedded_v1.h5ad"

# --- PCA ---
N_PCS = 50                          # 多算，用 elbow plot 决定实际使用数
N_PCS_USE = 30                      # 实际送入邻居图/Harmony 的 PC 数（经验值 20-40，胃全层组织建议 30）
                                    # N_PCS=50 只是"多算"用于 elbow 诊断，N_PCS_USE 才决定下游用多少

# --- 邻居图（支持 Scalar-or-Sweep）---
# 经验法则：k ≈ sqrt(n_cells)/2，最少 10 最多 100
#   - 稀有细胞类型关注 → 小 k（10-15）
#   - 大群概览 → 大 k（30-50）
#   - 最终标准：已知 marker（如 CD3）是否在 UMAP 上连续
N_NEIGHBORS = 15            # 单值直接跑 | 列表如 [10, 15, 20, 30] → 对比 UMAP 形态
METRIC = "cosine"                   # "euclidean" | "cosine"（cosine 对高维更稳健）

# --- UMAP ---
UMAP_MIN_DIST = 0.3                 # 0.1=紧密 0.5=分散（视觉参数，不影响分析）
UMAP_SPREAD   = 1.0

# --- 嵌入方法 ---
EMBEDDING_METHODS = ["pca", "harmony"]  # 可选追加: "scvi", "scanvi", "sccraft"

# --- 批次校正 key ---
BATCH_KEY = "source_dataset"        # 最粗粒度 | "sample_id" 细粒度

# --- Harmony（支持 Scalar-or-Sweep）---
HARMONY_THETA = 2.0                 # 单值直接跑 | 列表如 [1, 2, 3] → 对比校正强度
HARMONY_MAX_ITER = 20

# --- scVI ---
SCVI_N_LATENT    = 30
SCVI_N_LAYERS    = 2
SCVI_N_HIDDEN    = 128
SCVI_MAX_EPOCHS  = 200              # 生产 200-400；快测 50
SCVI_EARLY_STOPPING = True
SCVI_GENE_LIKELIHOOD = "zinb"

# --- 计算设备（device 自适应，见 ADR-0013）---
# "auto" 自动：CUDA GPU > (Mac)scVI/scANVI 用 CPU > CPU；可显式 "cuda"|"mps"|"cpu"
# 注意：scCRAFT 内部硬编码 CPU，本参数对 scCRAFT 无效
DEVICE = "auto"

# --- scANVI ---
SCANVI_N_EPOCHS = 100
SCANVI_LABEL_KEY = None             # None = 自动检测

# --- scCRAFT（需单独安装: git clone https://github.com/ch2343/scCRAFT && pip install .）---
SCCRAFT_RESOLUTION    = 0.5         # 低分辨率聚类系数
SCCRAFT_CLUSTER_METHOD = "Leiden"   # "Louvain"（快）或 "Leiden"（慢但更准）
SCCRAFT_EPOCHS        = 150         # batch 数 > 80 时降到 50
SCCRAFT_WARMUP_EPOCH  = 50          # 约为 epochs 的 1/3
SCCRAFT_D_COEF        = 0.2         # 判别器损失系数（越高批次混合越强）
SCCRAFT_KL_COEF       = 0.005       # KL 散度比例（batch effect 小时降到 0.0005 更好保留细胞身份）
SCCRAFT_N_TOP_GENES   = 2000        # scCRAFT 内部 HVG 选择数量

OUTPUT_VERSION = 1
RANDOM_SEED    = 42

# --- 跨 cell 状态变量预初始化（防止 cell 跳执行时下游 NameError）---
_counts_key = None
_counts_source = None
_converged = True

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_04", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 导入（scanpy 原生 API + 框架函数仅在真正需要时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import warnings

# 直接从 scorers 模块导入指标函数——无回调抽象，在 for 循环中直接调用
from scrna_integration.scorers import integration_metrics

# 抑制 scvi-tools PyTorch Lightning 弃用警告
warnings.filterwarnings("ignore", message=".*Lightning.*")
warnings.filterwarnings("ignore", message=".*The number of training batches.*")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")

# 确定 UMAP 着色列（全程复用）。
# sample_id——样本是否混合均匀（批次整合质量）
# BATCH_KEY——声明的批次变量是否被校正
# 细胞类型列——已知细胞类型是否被分离（生物学信号保留）
colour_columns = ["sample_id"]
if BATCH_KEY in adata.obs.columns and BATCH_KEY != "sample_id":
    colour_columns.append(BATCH_KEY)
for ct_cand in ["Celltypes_global", "cell_type", "cell_type_original"]:
    if ct_cand in adata.obs.columns:
        colour_columns.append(ct_cand)
        break
print(f"UMAP 着色列: {colour_columns}")

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包，见 ADR-0012）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 计算设备概览（device 自适应，见 ADR-0013）
from scrna_integration.platform import detect_device
_dev_overview = detect_device(prefer=DEVICE)
print(f"计算设备: {_dev_overview['device_str']}  ({_dev_overview['reason']})")

In [ ]:
# === Counts 来源发现（scVI/scANVI/DESeq2 的前置验证）===
# scVI/scANVI 依赖原始整数 counts 做负二项建模。小数或对数化的
# counts 会破坏模型假设，训练效果急剧下降。本 cell 自动发现可用的
# 原始 counts 来源，避免手动排查。
_counts_source = None
_counts_key = None  # 最终供 scVI 使用的 layer key

# 优先级 1: layers["counts"]——03 pipeline 标准产出
if "counts" in adata.layers:
    _sample = adata.layers["counts"][:500, :500]
    if sp.issparse(_sample):
        _sample = _sample.toarray()
    _is_int = np.allclose(_sample, np.round(_sample), atol=0.01)
    if _is_int:
        _counts_source = "layers['counts']"
        _counts_key = "counts"
        print(f"✓ 发现原始 counts: adata.layers['counts'] (整数验证通过)")
    else:
        print(f"⚠️ adata.layers['counts'] 存在但含非整数值——不可直接用于 scVI")

# 优先级 2: adata.raw.X（fallback——某些上游可能只存 raw 未复制到 layers）
if _counts_key is None and adata.raw is not None:
    _sample_raw = adata.raw.X[:500, :500]
    if sp.issparse(_sample_raw):
        _sample_raw = _sample_raw.toarray()
    _is_int_raw = np.allclose(_sample_raw, np.round(_sample_raw), atol=0.01)
    if _is_int_raw:
        print(f"✓ 发现原始 counts: adata.raw.X (整数验证通过)")
        if "counts" not in adata.layers:
            # layers 中无 counts 层——从 raw.X 创建（仅此一次）
            if adata.raw.X.shape[1] == adata.n_vars:
                adata.layers["counts"] = adata.raw.X.copy()
            else:
                _raw_var = adata.raw.var_names
                _current_var = adata.var_names
                _shared = _current_var.isin(_raw_var)
                if _shared.all():
                    _gene_to_idx = {g: i for i, g in enumerate(_raw_var)}
                    _raw_idx = [_gene_to_idx[g] for g in _current_var]
                    adata.layers["counts"] = adata.raw.X[:, _raw_idx].copy()
                    print(f"  raw.X 基因数({len(_raw_var)}) > 当前({adata.n_vars})，已对齐子集")
                else:
                    print(f"  ⚠️ raw.X 与当前 adata 基因名不完全匹配，无法自动对齐")
                    _is_int_raw = False  # 标记失败
            if "counts" in adata.layers:
                _counts_source = "adata.raw.X → layers['counts'] (stage 04 创建)"
                _counts_key = "counts"
                print(f"  已写入 layers['counts']（原始 layers 中不存在，从 raw.X 补建）")
        else:
            # layers["counts"] 已存在但上面整数检查没通过——说明 layers 的值有问题
            # 此处 raw 是整数，layers 不是——用 raw 覆盖？不，保守策略：不覆盖，警告
            _counts_source = "adata.raw.X (未写入 layers——layers['counts'] 已存在)"
            _counts_key = "counts"  # 仍用已有的 layers["counts"]
            print(f"  layers['counts'] 已存在（由上游 stage 03 产出），不覆盖")
    else:
        print(f"⚠️ adata.raw.X 存在但含非整数值")

# 两者都没有——scVI/scANVI 将无法运行
if _counts_key is None:
    print("❌ 未找到可用的原始整数 counts（layers['counts'] 和 adata.raw.X 均不可用）")
    print("   scVI/scANVI 将无法运行。请检查上游数据。")

# 重复存储检测：layers['counts'] 和 adata.raw.X 是否内容相同
if "counts" in adata.layers and adata.raw is not None:
    _raw_shape = adata.raw.X.shape
    _layers_shape = adata.layers["counts"].shape
    if _raw_shape[0] == _layers_shape[0]:
        # 快速比较：抽样 100 行对比
        _idx = np.random.choice(_raw_shape[0], size=min(100, _raw_shape[0]), replace=False)
        _raw_sample = adata.raw.X[_idx, :min(500, _raw_shape[1])]
        _lay_sample = adata.layers["counts"][_idx, :min(500, _layers_shape[1])]
        if sp.issparse(_raw_sample): _raw_sample = _raw_sample.toarray()
        if sp.issparse(_lay_sample): _lay_sample = _lay_sample.toarray()
        # 如果列数相同且内容一致——存在重复存储
        if _raw_shape[1] == _layers_shape[1] and np.allclose(_raw_sample, _lay_sample):
            _mem_mb = 0
            if sp.issparse(adata.layers["counts"]):
                _mem_mb = (adata.layers["counts"].data.nbytes + adata.layers["counts"].indices.nbytes + adata.layers["counts"].indptr.nbytes) / 1024**2
            else:
                _mem_mb = adata.layers["counts"].nbytes / 1024**2
            print(f"\n💡 adata.raw.X 与 layers['counts'] 内容一致（抽样验证）")
            print(f"   重复存储约 {_mem_mb:.0f} MB。如需节省内存，可删除 adata.raw:")
            print(f"   del adata.raw  # 释放 ~{_mem_mb:.0f} MB（scVI 只从 layers['counts'] 读取）")

if _counts_key:
    print(f"\n最终 scVI/scANVI 使用: adata.layers['{_counts_key}']")

In [ ]:
# === 批次规模诊断 ===
# 极端不平衡的批次规模（>10x）会导致 scVI 对小批次过拟合、
# Harmony 校正不足等问题。提前报告让 PI 知情。
if BATCH_KEY in adata.obs.columns:
    _batch_counts = adata.obs[BATCH_KEY].value_counts()
    print(f"\n===== 批次规模（{BATCH_KEY}）=====")
    for b, n in _batch_counts.items():
        print(f"  {b}: {n:,}")
    _ratio = _batch_counts.max() / _batch_counts.min()
    print(f"  最大/最小比: {_ratio:.1f}x")
    if _ratio > 10:
        print("  ⚠️ 批次规模差异 > 10 倍")
        print("  → scVI 可能对小批次过拟合，建议减小 batch_size 或对大批次 downsample")
    elif _ratio > 5:
        print("  △ 批次规模差异 5-10 倍，注意观察 scVI UMAP 中小批次是否过度分离")
    else:
        print("  ✓ 批次规模均衡")
else:
    print(f"\n===== 批次规模诊断（跳过——BATCH_KEY='{BATCH_KEY}' 不在 obs 列中）=====")

## PCA（基线）

在高可变基因（HVG）上做主成分分析。这是最简单的嵌入方法——
无批次校正、无深度模型。作为所有整合方法的基线对照。

In [ ]:
# 在高可变基因（HVG）上做主成分分析。
if "pca" in EMBEDDING_METHODS:
    print(f"\n===== PCA: n_comps={N_PCS} =====")
    sc.tl.pca(adata, n_comps=N_PCS, use_highly_variable=True,
              svd_solver="arpack", random_state=RANDOM_SEED)
    print(f"obsm['X_pca'] shape: {adata.obsm['X_pca'].shape}")
    var_explained = adata.uns['pca']['variance_ratio'].sum() * 100
    print(f"累计方差解释率 (前 {N_PCS} PCs): {var_explained:.1f}%")
else:
    print("PCA 跳过（不在 EMBEDDING_METHODS 中）")

In [ ]:
# --- PCA Elbow Plot：经验选择维度数 ---
# 方差解释率随 PC 数递减。拐点（elbow）之后的 PC 主要是噪声。
# 仅当 PCA 已计算时运行。
if "X_pca" in adata.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # 左图：方差解释率
    # scanpy >=1.10 移除了 ax= 参数，改用 plt.sca() 设置当前轴
    plt.sca(axes[0])
    sc.pl.pca_variance_ratio(adata, n_pcs=N_PCS, log=False, show=False)
    axes[0].set_title("PCA 方差解释率")

    # 右图：累积方差
    var_ratio = adata.uns["pca"]["variance_ratio"][:N_PCS]
    cumulative = np.cumsum(var_ratio)
    axes[1].plot(range(1, N_PCS + 1), cumulative, "o-", markersize=3)
    axes[1].axhline(0.9, color="red", linestyle="--", alpha=0.5, label="90% 累积方差")
    axes[1].set_xlabel("PC")
    axes[1].set_ylabel("累积方差解释率")
    axes[1].set_title("累积方差（红线=90%）")
    axes[1].legend()

    plt.tight_layout(); plt.show()

    # 自动检测 elbow（二阶差分最大处）
    diff2 = np.diff(var_ratio, 2)
    elbow_pc = np.argmax(np.abs(diff2)) + 2  # +2 因为两次 diff 丢 2 个点
    print(f"建议 N_PCS: ~{elbow_pc}（二阶差分拐点）")
    pc90 = np.searchsorted(cumulative, 0.9) + 1
    print(f"   当前使用 N_PCS={N_PCS}，90% 累积方差在 PC {pc90}")
    # N_PCS_USE 实际应用提醒：elbow 用于诊断整个 PCA 空间，但下游只截用前 N_PCS_USE 维
    print(f"   实际使用: N_PCS_USE={N_PCS_USE}（送入邻居图/Harmony）")
    if N_PCS_USE < elbow_pc:
        print(f"   ⚠️ N_PCS_USE({N_PCS_USE}) < elbow 建议({elbow_pc})——可考虑增大")
    elif N_PCS_USE > pc90:
        print(f"   ⚠️ N_PCS_USE({N_PCS_USE}) > 90%方差({pc90})——可能引入噪音维度")
else:
    print("PCA 未计算，跳过 Elbow Plot。")

## PCA Loading 分析

检查前 3 个主成分由哪些基因驱动。如果技术基因（MT- / RPS / RPL）在 top loading 中
占据显著位置，说明 03 的 HVG 排除列表并未真正生效——这些技术基因仍通过 PCA 传递到
下游，需要回 03 调整排除参数或启用 `regress_out`。


In [ ]:
if "PCs" not in adata.varm:
    print("PCA 未运行（不在 EMBEDDING_METHODS 中），跳过 loading 分析")
else:
    # PCA loading 分析：前 3 个 PC 由哪些基因驱动？
    # 如果技术基因（MT-/RPS/RPL）主导 → 说明 03 排除列表没生效，需回 03 调整
    print("===== PCA Loading 分析（前 3 个 PC 的 top 10 genes）=====")
    loadings = pd.DataFrame(
        adata.varm["PCs"][:, :3],
        index=adata.var_names,
        columns=[f"PC{i+1}" for i in range(3)]
    )
    for pc in loadings.columns:
        top_pos = loadings[pc].nlargest(5)
        top_neg = loadings[pc].nsmallest(5)
        print(f"\\n{pc} top positive: {', '.join(top_pos.index)}")
        print(f"{pc} top negative: {', '.join(top_neg.index)}")

    # 检查技术基因是否主导 PC1-3
    tech_genes_in_top = []
    for pc in loadings.columns:
        top10 = loadings[pc].abs().nlargest(10).index
        mt_count = sum(g.startswith("MT-") for g in top10)
        ribo_count = sum(g.startswith(("RPS", "RPL")) for g in top10)
        if mt_count >= 3 or ribo_count >= 3:
            tech_genes_in_top.append(f"{pc}: MT={mt_count}, Ribo={ribo_count}")
    if tech_genes_in_top:
        print(f"\\n⚠️ 技术基因主导 PCA: {tech_genes_in_top}")
        print("  → 建议回 03 启用 EXCLUDE_MT_FROM_HVG / REGRESS_OUT")
    else:
        print("\\n✓ 前 3 个 PC 未被技术基因主导")

    # N_NEIGHBORS 经验建议
    _suggested_k = max(10, min(100, int(np.sqrt(adata.n_obs) / 2)))
    print(f"📊 经验建议 N_NEIGHBORS: {_suggested_k}（sqrt({adata.n_obs:,})/2，当前使用 k={N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]}）")

    # --- Stress PC 检测：解离诱导应激基因是否主导某个 PC ---
    # 组织解离过程会激活即时早期基因（IEG）和热休克蛋白，这些"应激
    # 特征"如果集中在某个 PC 上主导，会在下游 UMAP 中形成与生物学
    # 无关的簇。提前发现，可在 stage 03 做 regress_out 或在 Harmony
    # 中对该 PC 加强校正。
    _stress_genes = {"FOS", "JUN", "JUNB", "JUND", "ATF3", "EGR1", "HSPA1A", "HSPA1B",
                     "DUSP1", "ZFP36", "IER2", "NR4A1", "FOSB", "KLF2", "KLF4"}
    print(f"\n--- Stress PC 检测（解离应激基因）---")
    _stress_pcs = []
    for pc_idx in range(min(5, N_PCS)):
        _loading = adata.varm["PCs"][:, pc_idx]
        _top20_idx = np.argsort(np.abs(_loading))[-20:]
        _top20_genes = set(str(g).upper() for g in adata.var_names[_top20_idx])
        _overlap = _top20_genes & _stress_genes
        if len(_overlap) >= 3:
            _stress_pcs.append(pc_idx + 1)
            print(f"  ⚠️ PC{pc_idx+1} 含 {len(_overlap)} 个应激基因: {sorted(_overlap)}")
    if _stress_pcs:
        print(f"  → 如 UMAP 上观察到应激驱动的独立 cluster，可考虑:")
        print(f"    (a) 在 stage 03 对 S_score/G2M_score + 应激评分做 regress_out")
        print(f"    (b) 增大 Harmony theta 让其在这些 PC 上更强校正")
        print(f"    (c) 排除这些 PC（高级用法，需谨慎）")
    else:
        print(f"  ✓ 前 5 个 PC 无显著应激基因主导")

### PCA UMAP（即时查看）

PCA 是最简单的嵌入——无批次校正。作为后续方法的**基线对照**。

**怎么看**：
- 不同样本的细胞是否明显分离？→ 批次效应强，需要校正
- 已知细胞类型是否大致分开？→ 生物学信号在 PCA 空间是否可辨

对比后续 Harmony / scVI 的 UMAP，判断批次校正是否改善了混合。

In [ ]:
# PCA UMAP —— 即时查看嵌入质量
embed_key = "X_pca"
if embed_key in adata.obsm:
    print(f"\n===== PCA UMAP =====")
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_pcs=min(N_PCS_USE, adata.obsm[embed_key].shape[1]),
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key，后续指标计算可复用。
    adata.obsm["X_umap_pca"] = adata.obsm["X_umap"].copy()

    # 三着色出图：样本、批次、细胞类型（如有）。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"PCA - {colour}",
                       frameon=False, save="_pca_{colour}.png")
            src = f"figures/umap_pca_{colour}.png"
            dst = f"results/figures/04_umap_pca_{colour}.png"
            if os.path.exists(src):
                os.rename(src, dst)
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"PCA - doublet_score",
                   frameon=False, save="_pca_doublet_score.png")
        src = "figures/umap_pca_doublet_score.png"
        dst = "results/figures/04_umap_pca_doublet_score.png"
        if os.path.exists(src):
            os.rename(src, dst)
        plt.close("all")

    print("PCA UMAP 完成。")
else:
    print("PCA 嵌入不存在，跳过 UMAP。")

## Harmony

Harmony 通过迭代校正 PCA 嵌入来整合批次。速度快、确定性算法，
在单细胞分析管线中广泛使用。与 PCA 共用相同 `N_PCS` 参数，
batch key 来自 PARAMS 单元格。

In [ ]:
# Harmony 在 PCA 嵌入上做批次整合。
# 注意：直接使用 harmonypy（而非 sc.external.pp.harmony_integrate），
# 因为 harmonypy >= 2.0.0 把 Z_corr 方向从 (dims, cells) 改为
# (cells, dims)；scanpy wrapper 仍假设旧布局做 .T 导致 shape 不匹配。
# 故本 cell 直接调用 harmonypy 避免 scanpy wrapper 的兼容问题。
if "harmony" in EMBEDDING_METHODS:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 Harmony")
    else:
        print(f"\n===== Harmony: batch_key='{BATCH_KEY}' =====")
        import harmonypy

        # 只喂前 N_PCS_USE 个 PC 给 Harmony——后端 PC 含高噪声，会稀释校正信号
        ho = harmonypy.run_harmony(
            adata.obsm["X_pca"][:, :N_PCS_USE],
            adata.obs,
            BATCH_KEY,
            max_iter_harmony=HARMONY_MAX_ITER,
        )
        # ho.Z_corr 在 harmonypy >= 2.0.0 中已经是 (n_cells, n_pcs) 方向。
        # 输出也是 N_PCS_USE 维——只保留前端有信号维度
        assert ho.Z_corr.shape[1] == N_PCS_USE, (
            f"Harmony Z_corr 维度不符: {ho.Z_corr.shape[1]} != N_PCS_USE({N_PCS_USE})")
        adata.obsm["X_pca_harmony"] = ho.Z_corr
        print(f"obsm['X_pca_harmony'] shape: {adata.obsm['X_pca_harmony'].shape}")

        # --- Harmony 收敛检查（不同版本 harmonypy API 兼容）---
        if hasattr(ho, 'check_convergence') and callable(ho.check_convergence):
            _converged = ho.check_convergence()
        elif hasattr(ho, 'converged'):
            _c = ho.converged
            _converged = _c() if callable(_c) else _c
        else:
            # harmonypy 某些版本没有公开收敛检查端口，默认假设通过
            _converged = True
        if not _converged:
            print(f"⚠️ Harmony 未在 {HARMONY_MAX_ITER} 轮内收敛")
            print(f"  → 考虑: 增大 HARMONY_MAX_ITER 或降低 HARMONY_THETA")
        else:
            print(f"✓ Harmony 收敛（max_iter={HARMONY_MAX_ITER}）")
else:
    print("Harmony 跳过（不在 EMBEDDING_METHODS 中）")

### Harmony UMAP（即时查看）

Harmony 在 PCA 空间上迭代校正批次效应。速度快、确定性算法。

**怎么看**：
- 不同样本的细胞是否**混合均匀**？→ 好
- 混合是否过度抹平了生物学差异？→ 警惕过度校正
- 与上方 PCA UMAP 对比：混合改善了，但细胞类型分离是否保留？

如果 Harmony 效果不理想（样本仍分离、或过度混合），可尝试 scVI。

In [ ]:
# Harmony UMAP —— 即时查看批次校正效果
embed_key = "X_pca_harmony"
if embed_key in adata.obsm:
    print(f"\n===== Harmony UMAP =====")
    sc.pp.neighbors(adata, use_rep=embed_key,
                    # Harmony 输出 = N_PCS_USE 维，全部使用
                    n_pcs=min(N_PCS_USE, adata.obsm[embed_key].shape[1]),
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_pca_harmony"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"Harmony - {colour}",
                       frameon=False, save="_pca_harmony_{colour}.png")
            src = f"figures/umap_pca_harmony_{colour}.png"
            dst = f"results/figures/04_umap_pca_harmony_{colour}.png"
            if os.path.exists(src):
                os.rename(src, dst)
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"Harmony - doublet_score",
                   frameon=False, save="_pca_harmony_doublet_score.png")
        src = "figures/umap_pca_harmony_doublet_score.png"
        dst = "results/figures/04_umap_pca_harmony_doublet_score.png"
        if os.path.exists(src):
            os.rename(src, dst)
        plt.close("all")

    print("Harmony UMAP 完成。")
else:
    print("Harmony 嵌入不存在，跳过 UMAP。")

## scVI（从头训练）

scVI（单细胞变分推断）学习一个显式将批次效应建模为干扰变量的潜表示。
需要原始 counts——使用 `adata.layers['counts']`（03 保留的原始计数）。

**快速验证说明**：上方 `SCVI_MAX_EPOCHS` 设得较低以加速管线调试。
生产质量嵌入建议增加到 200-400 epochs。监控训练损失曲线确认收敛。

**为什么至少需要 20 epochs？** scVI 的变分推断需要足够迭代才能收敛到
有意义的潜空间。太少的 epochs 会导致嵌入质量差、下游 UMAP 和聚类
无法反映真实的生物学结构。200-400 epochs 是 scVI 论文和社区的推荐值。

scVI 设置步骤：
1. `setup_anndata`——声明哪些 layer/列存放 counts、batch 等
2. `SCVI(adata)`——实例化模型
3. `.train()`——训练（含早停）
4. `.get_latent_representation()`——提取嵌入到 `obsm["X_scVI"]`

In [ ]:
# scVI：setup_anndata + 训练 + 提取潜表示。
if "scvi" in EMBEDDING_METHODS:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 scVI")
    elif _counts_key is None:
        print("WARNING: 无可用的原始 counts——scVI 跳过。请检查上游数据。")
    else:
        print(f"\n===== scVI: max_epochs={SCVI_MAX_EPOCHS}, n_latent={SCVI_N_LATENT} =====")
        import scvi

        _dev = detect_device(prefer=DEVICE, for_method="scvi")
        print(f"scVI 设备: {_dev['device_str']}（{_dev['reason']}）")

        scvi.model.SCVI.setup_anndata(
            adata,
            layer=_counts_key,  # 使用 counts 发现 cell 的动态结果
            batch_key=BATCH_KEY,
        )

        model = scvi.model.SCVI(
            adata,
            n_layers=SCVI_N_LAYERS,
            n_latent=SCVI_N_LATENT,
            n_hidden=SCVI_N_HIDDEN,
            gene_likelihood="zinb",
            use_layer_norm="both",
            use_batch_norm="none",
        )

        print(f"训练 scVI (max_epochs={SCVI_MAX_EPOCHS})...")
        model.train(
            max_epochs=SCVI_MAX_EPOCHS,
            accelerator=_dev["accelerator"],
            devices=_dev["devices"],
            early_stopping=True,
            early_stopping_patience=5,
            plan_kwargs={"lr": 1e-3},
            check_val_every_n_epoch=1,
        )

        adata.obsm["X_scVI"] = model.get_latent_representation()
        print(f"obsm['X_scVI'] shape: {adata.obsm['X_scVI'].shape}")

        # --- scVI 训练收敛诊断 ---
        if hasattr(model, 'history') and "elbo_train" in model.history:
            history_df = model.history["elbo_train"]
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.plot(history_df.index, history_df.values.flatten(), color="steelblue")
            ax.set_xlabel("Epoch"); ax.set_ylabel("ELBO (train)")
            ax.set_title("scVI 训练收敛曲线")
            last10 = history_df.values.flatten()[-10:]
            cv = np.std(last10) / abs(np.mean(last10)) if np.mean(last10) != 0 else 999
            if cv < 0.01:
                print(f"✓ scVI 已收敛（最后 10 epoch CV={cv:.4f}）")
            else:
                print(f"⚠️ scVI 可能未收敛（CV={cv:.4f} > 0.01），建议增大 SCVI_MAX_EPOCHS")
            plt.tight_layout(); plt.show()

        print("scVI 完成。")
else:
    print("scVI 跳过（不在 EMBEDDING_METHODS 中）")

### scVI UMAP（即时查看）

scVI 通过变分推断学习将批次效应建模为干扰变量的潜表示。
相比 Harmony，能捕捉更复杂的批次-生物学交互。

**怎么看**：
- 样本混合是否优于 PCA 基线？是否优于 Harmony？
- 细胞类型分离是否清晰？
- 训练损失曲线（scVI 自动输出）是否收敛？未收敛 → 增加 `SCVI_MAX_EPOCHS` 重跑

scVI 训练时间较长。如果 UMAP 质量与 Harmony 相当或更差，
且 Harmony 已满足需求，后续可优先使用 Harmony（速度优势）。

In [ ]:
# scVI UMAP —— 即时查看潜空间质量
embed_key = "X_scVI"
if embed_key in adata.obsm:
    print(f"\n===== scVI UMAP =====")
    sc.pp.neighbors(adata, use_rep=embed_key,
                    # scVI 潜空间为 SCVI_N_LATENT 维，本身低维不需截断
                    n_pcs=min(SCVI_N_LATENT, adata.obsm[embed_key].shape[1]),
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scVI - {colour}",
                       frameon=False, save="_scVI_{colour}.png")
            src = f"figures/umap_scVI_{colour}.png"
            dst = f"results/figures/04_umap_scVI_{colour}.png"
            if os.path.exists(src):
                os.rename(src, dst)
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scVI - doublet_score",
                   frameon=False, save="_scVI_doublet_score.png")
        src = "figures/umap_scVI_doublet_score.png"
        dst = "results/figures/04_umap_scVI_doublet_score.png"
        if os.path.exists(src):
            os.rename(src, dst)
        plt.close("all")

    print("scVI UMAP 完成。")
else:
    print("scVI 嵌入不存在，跳过 UMAP。")

## （可选）cellxgene_census 预训练 scVI

使用 CZ CELLxGENE Census 预训练 scVI 模型对当前数据集进行嵌入。
该预训练模型将新细胞映射到从 CELLxGENE 语料库数百万细胞中学到的
潜空间中——无需本地训练。

**前置条件**：Census 必须托管了该组织（示例数据中为 gastric mucosa）
的 scVI 模型。可在 cellxgene.cziscience.com 查询可用性。

启用后，下方 cell 将：
1. 从 Census 下载预训练模型。
2. 通过 mygene 对齐基因符号。
3. 调用 `prepare_query_anndata` 和 `load_query_data`。
4. 提取潜表示到 `obsm["X_scVI_census"]`。

该 cell 默认**被注释掉**，因为 Census 作为重量级依赖增加安装负担，
且仅在组织存在优质模型时才有效。

In [ ]:
# # === cellxgene_census 预训练 scVI（已注释） ===
# # 前置条件：Census 存在该组织的预训练 scVI 模型。
# # 启用此 cell：取消下方注释，然后在 PARAMS 中添加
# # "cellxgene_census" 到 EMBEDDING_METHODS。
#
# # import cellxgene_census
# # census = cellxgene_census.open_soma(census_version="latest")
# # model = cellxgene_census.download_source_h5ad(
# #     ORGANISM, layer="scvi", tissue=TISSUE,
# # )
# # # 基因对齐（参见 legacy-GCPL/04_dimensionality_reduction.ipynb）
# # adata.obsm["X_scVI_census"] = model.get_latent_representation(adata)
# # print("cellxgene_census scVI 完成。")
#
# print("cellxgene_census 预训练 scVI cell 已注释。"
#       "Census 存在该组织模型时取消注释。")

## （可选）scANVI 标签迁移嵌入

scANVI（single-cell ANnotation Variational Inference，单细胞注释变分推断）
在 scVI 基础上扩展了细胞类型分类头。其产出的潜空间兼具批次校正与
细胞类型感知能力——当存在带标注的参考数据集且细胞类型在批次间
保守时尤为有用。

**前置条件**：`adata.obs` 中必须包含可用的细胞类型标签列。
本 cell 自动检测常见标签列名：
`cell_type_original_*`、`Celltypes_global`、`cell_type`、`label`。
若均未找到，则优雅跳过 scANVI 并输出日志提示。

In [ ]:
# scANVI：scVI 的半监督变体，利用细胞类型标签。
# 无可用标签时优雅跳过并给出说明。
if "scanvi" in EMBEDDING_METHODS:
    print("\n===== scANVI: 检测细胞类型标签列 =====")

    if _counts_key is None:
        print("scANVI 跳过: 无可用的原始 counts。请检查上游数据。")
    else:
        # 自动检测标签列：优先原始作者标注，其次常见列名。
        label_col = None
        for pattern in ["cell_type_original_", "Celltypes_global", "cell_type", "label",
                        "Detailed_Cell_Type", "Global_cluster_selected"]:
            for col in adata.obs.columns:
                if pattern in col:
                    label_col = col
                    break
            if label_col is not None:
                break

        if label_col is None:
            print("scANVI 跳过: 未在 adata.obs 中找到细胞类型标签列。")
            print("  可用 obs 列:", list(adata.obs.columns)[:10], "...")
            print("  启用 scANVI: 在运行前向 adata.obs 添加标签列。")
        else:
            n_labels = adata.obs[label_col].nunique()
            n_nan = adata.obs[label_col].isna().sum()
            print(f"找到标签列: '{label_col}' ({n_labels} unique, {n_nan} NaN)")

            if n_labels < 2:
                print(f"scANVI 跳过: '{label_col}' 的 unique 非 NaN 值 < 2。")
            else:
                import scvi
                from scvi.model import SCANVI

                _dev = detect_device(prefer=DEVICE, for_method="scanvi")
                print(f"scANVI 设备: {_dev['device_str']}（{_dev['reason']}）")

                scvi.model.SCVI.setup_anndata(
                    adata, layer=_counts_key, batch_key=BATCH_KEY,
                    labels_key=label_col,
                )
                scvi_model = scvi.model.SCVI(
                    adata, n_layers=SCVI_N_LAYERS, n_latent=SCVI_N_LATENT,
                    n_hidden=SCVI_N_HIDDEN,
                    gene_likelihood="zinb", use_layer_norm="both", use_batch_norm="none",
                )
                scvi_model.train(max_epochs=SCVI_MAX_EPOCHS,
                                 accelerator=_dev["accelerator"], devices=_dev["devices"],
                                 early_stopping=True, early_stopping_patience=5,
                                 plan_kwargs={"lr": 1e-3})

                scanvi_model = SCANVI.from_scvi_model(
                    scvi_model,
                    unlabeled_category="Unknown",
                    labels_key=label_col,
                )
                n_ep = max(5, SCVI_MAX_EPOCHS // 2)
                print(f"训练 scANVI (max_epochs={n_ep})...")
                scanvi_model.train(
                    max_epochs=n_ep,
                    accelerator=_dev["accelerator"],
                    devices=_dev["devices"],
                    early_stopping=True,
                    early_stopping_patience=3,
                )

                adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
                print(f"obsm['X_scANVI'] shape: {adata.obsm['X_scANVI'].shape}")
                print("scANVI 完成。")
else:
    print("scANVI 跳过（不在 EMBEDDING_METHODS 中）")

### scANVI UMAP（即时查看）

scANVI 在 scVI 基础上融入细胞类型标签，产出的潜空间**同时感知批次校正与细胞类型**。

**怎么看**：
- 与上方 scVI UMAP 对比：细胞类型分离是否更清晰？
- 样本混合是否维持？
- 如果标签质量差或覆盖不全，scANVI 可能不如纯 scVI——对比后择优

In [ ]:
# scANVI UMAP —— 即时查看标签感知潜空间（仅当 scANVI 已运行）
embed_key = "X_scANVI"
if embed_key in adata.obsm:
    print(f"\n===== scANVI UMAP =====")
    sc.pp.neighbors(adata, use_rep=embed_key,
                    # scANVI 潜空间为 SCVI_N_LATENT 维，本身低维不需截断
                    n_pcs=min(SCVI_N_LATENT, adata.obsm[embed_key].shape[1]),
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scANVI"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scANVI - {colour}",
                       frameon=False, save="_scANVI_{colour}.png")
            src = f"figures/umap_scANVI_{colour}.png"
            dst = f"results/figures/04_umap_scANVI_{colour}.png"
            if os.path.exists(src):
                os.rename(src, dst)
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scANVI - doublet_score",
                   frameon=False, save="_scANVI_doublet_score.png")
        src = "figures/umap_scANVI_doublet_score.png"
        dst = "results/figures/04_umap_scANVI_doublet_score.png"
        if os.path.exists(src):
            os.rename(src, dst)
        plt.close("all")

    print("scANVI UMAP 完成。")
else:
    print("scANVI 嵌入不存在（未运行或无可用的细胞类型标签），跳过 UMAP。")

## scCRAFT（可选嵌入方法，anchor-free 批次整合）

scCRAFT 是一种 anchor-free 的批次校正整合方法（VAE + 判别器 + 部分拓扑结构）。
与 scVI 类似从原始 counts 学习潜空间，但用对抗训练增强批次混合。

**安装**（非 PyPI 包，需从 GitHub 源码安装）:
```bash
git clone https://github.com/ch2343/scCRAFT && cd scCRAFT && pip install .
```

**重要**: scCRAFT 内部会对传入的 AnnData 做 normalize + HVG 子集化（会修改对象）。因此本 notebook 在**独立的 counts 副本**上运行 scCRAFT，绝不触碰主 adata——结果通过 obs_names 对齐写回 `adata.obsm["X_scCRAFT"]`，主 adata 的 X / layers / 其他 obsm 完全不受影响。

In [ ]:
# scCRAFT：在独立 counts 副本上训练，结果对齐写回主 adata。
# 关键隔离：scCRAFT 内部会 normalize + log1p + 子集化到 HVG，
# 这些突变必须在独立副本上完成，绝不触碰主 adata。
if "sccraft" in EMBEDDING_METHODS:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中；跳过 scCRAFT")
    elif _counts_key is None:
        print("WARNING: 无可用的原始 counts——scCRAFT 跳过。")
    else:
        try:
            import scCRAFT
            from scCRAFT.model import (
                multi_resolution_cluster,
                train_integration_model,
                obtain_embeddings,
            )
            _sccraft_available = True
        except ImportError:
            print("scCRAFT 未安装，跳过。")
            print("   安装: git clone https://github.com/ch2343/scCRAFT && cd scCRAFT && pip install .")
            _sccraft_available = False

        if _sccraft_available:
            print(f"\n===== scCRAFT: epochs={SCCRAFT_EPOCHS}, d_coef={SCCRAFT_D_COEF}, kl_coef={SCCRAFT_KL_COEF} =====")
            import anndata as _ad

            # 构建独立的 counts AnnData——绝不 mutate 主 adata
            # scCRAFT 内部会 normalize + log1p + HVG 子集化，必须在副本上做
            _sccraft_adata = _ad.AnnData(
                X=adata.layers[_counts_key].copy(),
                obs=adata.obs[[BATCH_KEY]].copy(),
                var=adata.var[[]].copy(),
            )
            _sccraft_adata.obs_names = adata.obs_names.copy()
            _sccraft_adata.var_names = adata.var_names.copy()
            print(f"  独立副本: {_sccraft_adata.n_obs} cells x {_sccraft_adata.n_vars} genes")

            # scCRAFT 标准预处理（在副本上——不影响主 adata）
            _sccraft_adata.raw = _sccraft_adata.copy()
            sc.pp.filter_genes(_sccraft_adata, min_cells=5)
            sc.pp.normalize_per_cell(_sccraft_adata, counts_per_cell_after=1e4)
            sc.pp.log1p(_sccraft_adata)
            sc.pp.highly_variable_genes(_sccraft_adata, n_top_genes=SCCRAFT_N_TOP_GENES,
                                        batch_key=BATCH_KEY)
            _sccraft_adata = _sccraft_adata[:, _sccraft_adata.var["highly_variable"]].copy()
            print(f"  HVG 子集后: {_sccraft_adata.n_vars} genes")

            try:
                # scCRAFT 训练流程
                multi_resolution_cluster(_sccraft_adata, resolution1=SCCRAFT_RESOLUTION,
                                         method=SCCRAFT_CLUSTER_METHOD)
                # 注：scCRAFT 内部硬编码 CPU（self.device='cpu'），DEVICE 参数对其无效。
                # Mac/Linux 均走 CPU，跨平台一致。见 ADR-0013。
                _dev_sccraft = detect_device(prefer=DEVICE, for_method="sccraft")
                print(f"scCRAFT 设备: {_dev_sccraft['device_str']}（{_dev_sccraft['reason']}）")
                _vae = train_integration_model(
                    _sccraft_adata, batch_key=BATCH_KEY,
                    epochs=SCCRAFT_EPOCHS, d_coef=SCCRAFT_D_COEF,
                    kl_coef=SCCRAFT_KL_COEF, warmup_epoch=SCCRAFT_WARMUP_EPOCH,
                )
                obtain_embeddings(_sccraft_adata, _vae)

                # 对齐写回主 adata（按 obs_names 保证行顺序一致）
                if "X_scCRAFT" in _sccraft_adata.obsm:
                    _embed_df = pd.DataFrame(
                        _sccraft_adata.obsm["X_scCRAFT"],
                        index=_sccraft_adata.obs_names,
                    )
                    # 按主 adata 的 obs_names 顺序对齐
                    _embed_aligned = _embed_df.reindex(adata.obs_names).values
                    adata.obsm["X_scCRAFT"] = _embed_aligned.astype(np.float32)
                    print(f"  obsm['X_scCRAFT'] shape: {adata.obsm['X_scCRAFT'].shape}")
                    # 验证无 NaN（对齐失败会产生 NaN）
                    _n_nan = np.isnan(adata.obsm["X_scCRAFT"]).any(axis=1).sum()
                    if _n_nan > 0:
                        print(f"  ⚠️ {_n_nan} 个细胞的 scCRAFT 嵌入为 NaN（obs_names 对齐失败）")
                        print(f"  → 删除 X_scCRAFT（避免含 NaN 数据流入下游 UMAP/指标/推荐）")
                        del adata.obsm["X_scCRAFT"]
                else:
                    print("WARNING: scCRAFT 未产出 X_scCRAFT，检查训练是否成功")
            except Exception as e:
                print(f"WARNING: scCRAFT 训练失败: {e}")
                import traceback
                traceback.print_exc()
            finally:
                # 清理副本释放内存
                del _sccraft_adata
                import gc as _gc
                _gc.collect()
            print("scCRAFT 完成。")
else:
    print("scCRAFT 跳过（不在 EMBEDDING_METHODS 中）")

In [ ]:
# scCRAFT UMAP —— 即时查看潜空间质量
embed_key = "X_scCRAFT"
if embed_key in adata.obsm:
    print(f"\n===== scCRAFT UMAP =====")
    _sc_dim = adata.obsm[embed_key].shape[1]
    # 独立 UMAP 用 N_NEIGHBORS 仅作可视化初判；定量整合指标见后文整合指标对比 cell（用自适应 k）
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_pcs=min(_sc_dim, adata.obsm[embed_key].shape[1]),
                    n_neighbors=N_NEIGHBORS, random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scCRAFT"] = adata.obsm["X_umap"].copy()

    # 三着色出图：样本、批次、细胞类型（如有）。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scCRAFT - {colour}",
                       frameon=False, save="_scCRAFT_{colour}.png")
            src = f"figures/umap_scCRAFT_{colour}.png"
            dst = f"results/figures/04_umap_scCRAFT_{colour}.png"
            if os.path.exists(src):
                os.rename(src, dst)
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scCRAFT - doublet_score",
                   frameon=False, save="_scCRAFT_doublet_score.png")
        src = "figures/umap_scCRAFT_doublet_score.png"
        dst = "results/figures/04_umap_scCRAFT_doublet_score.png"
        if os.path.exists(src):
            os.rename(src, dst)
        plt.close("all")

    print("scCRAFT UMAP 完成。")
else:
    print("scCRAFT UMAP 跳过（X_scCRAFT 不存在）")

In [ ]:
# --- Scalar-or-Sweep：N_NEIGHBORS ---
# 不同 k 值影响 UMAP 局部-全局结构平衡。
# 列表模式：对每个 k 独立计算 neighbors + UMAP，grid 对比形态差异。
# 单值模式：跳过 sweep，下游直接使用主流程中已计算的 neighbors。
_k_values = N_NEIGHBORS if isinstance(N_NEIGHBORS, list) else [N_NEIGHBORS]

if len(_k_values) > 1:
    # 选择 sweep 用的嵌入：优先 X_pca_harmony，其次 X_pca
    sweep_rep = None
    for candidate in ["X_pca_harmony", "X_pca"]:
        if candidate in adata.obsm:
            sweep_rep = candidate
            break
    if sweep_rep is None:
        print("N_NEIGHBORS sweep 跳过：无可用的 obsm 嵌入。")
    else:
        print(f"N_NEIGHBORS sweep: {_k_values} on {sweep_rep}")

        n_cols = min(len(_k_values), 4)
        n_rows = (len(_k_values) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
        # 统一 flatten：单 subplot 时 axes 不是 ndarray
        if n_rows == 1 and n_cols == 1:
            axes_flat = [axes]
        else:
            axes_flat = axes.flatten()

        for i, k in enumerate(_k_values):
            sc.pp.neighbors(adata, use_rep=sweep_rep, n_neighbors=k,
                            metric=METRIC, random_state=RANDOM_SEED)
            sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                       random_state=RANDOM_SEED)
            adata.obsm[f"X_umap_k{k}"] = adata.obsm["X_umap"].copy()
            color_col = BATCH_KEY if BATCH_KEY in adata.obs.columns else colour_columns[0]
            sc.pl.umap(adata, color=color_col, ax=axes_flat[i], show=False,
                       title=f"k={k}")

        # 隐藏多余 subplot
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)

        plt.suptitle(f"N_NEIGHBORS sweep（{sweep_rep}，不同 k 下 UMAP 形态）")
        plt.tight_layout(); plt.show()

        # 恢复为列表最后一个 k 的 neighbors + UMAP
        sc.pp.neighbors(adata, use_rep=sweep_rep, n_neighbors=_k_values[-1],
                        metric=METRIC, random_state=RANDOM_SEED)
        sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                   random_state=RANDOM_SEED)
        print(f"N_NEIGHBORS sweep 完成。最终使用 k={_k_values[-1]} 继续下游。")
else:
    print(f"N_NEIGHBORS 单值模式 (k={_k_values[0]})，跳过 sweep。")

In [ ]:
# --- Scalar-or-Sweep：HARMONY_THETA ---
# 不同 theta 值控制 Harmony 批次校正强度。
# theta 越大 = 校正越强（样本混合越好，但可能过度抹平生物学差异）。
# 列表模式：对每个 theta 重新运行 Harmony + UMAP，grid 对比。
# 单值模式：跳过 sweep。
_t_values = HARMONY_THETA if isinstance(HARMONY_THETA, list) else [HARMONY_THETA]

if len(_t_values) > 1 and "X_pca" in adata.obsm:
    print(f"HARMONY_THETA sweep: {_t_values}")
    import harmonypy

    n_cols = min(len(_t_values), 4)
    n_rows = (len(_t_values) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes_flat = [axes]
    else:
        axes_flat = axes.flatten()

    _use_k = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else _k_values[-1]

    for i, theta in enumerate(_t_values):
        # 只喂前 N_PCS_USE 个 PC——与主 Harmony cell 保持一致
        ho = harmonypy.run_harmony(
            adata.obsm["X_pca"][:, :N_PCS_USE], adata.obs, BATCH_KEY,
            theta=theta, max_iter_harmony=HARMONY_MAX_ITER,
        )
        adata.obsm[f"X_pca_harmony_t{theta}"] = ho.Z_corr

        sc.pp.neighbors(adata, use_rep=f"X_pca_harmony_t{theta}",
                        n_neighbors=_use_k, metric=METRIC,
                        n_pcs=N_PCS_USE, random_state=RANDOM_SEED)
        sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                   random_state=RANDOM_SEED)

        color_col = BATCH_KEY if BATCH_KEY in adata.obs.columns else colour_columns[0]
        sc.pl.umap(adata, color=color_col, ax=axes_flat[i], show=False,
                   title=f"theta={theta}")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle("HARMONY_THETA sweep（不同 theta 下批次混合 vs 生物学保留）")
    plt.tight_layout(); plt.show()

    # 恢复为主 HARMONY_THETA（列表最后一个值）的嵌入
    adata.obsm["X_pca_harmony"] = adata.obsm[f"X_pca_harmony_t{_t_values[-1]}"].copy()
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=_use_k,
                    metric=METRIC, n_pcs=N_PCS_USE, random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)
    print(f"HARMONY_THETA sweep 完成。最终使用 theta={_t_values[-1]}。")
elif "harmony" in EMBEDDING_METHODS and "X_pca" in adata.obsm:
    print(f"HARMONY_THETA 单值模式 (theta={_t_values[0]})，跳过 sweep。")
else:
    print("Harmony 未启用或 PCA 未计算，跳过 HARMONY_THETA sweep。")

In [ ]:
# === Marker 基因 UMAP 叠印（评估嵌入空间的生物信号保留）===
MARKER_GENES_UMAP = ["EPCAM", "PTPRC", "VIM", "ATP4A", "PGA3", "CDX2", "MKI67", "CD3D"]

# 使用最后跑完的嵌入的 UMAP
_last_umap = None
for _k in ["X_umap_scVI", "X_umap_pca_harmony", "X_umap_scANVI", "X_umap_pca"]:
    if _k in adata.obsm:
        _last_umap = _k
        break

if _last_umap:
    # 临时设为默认 UMAP（sc.pl.umap 使用 X_umap）
    _backup_umap = adata.obsm.get("X_umap", None)
    adata.obsm["X_umap"] = adata.obsm[_last_umap]
    
    _avail = [g for g in MARKER_GENES_UMAP if g in adata.var_names]
    if _avail:
        print(f"Marker 基因 UMAP 叠印（{_last_umap} 空间，{len(_avail)} 基因）")
        sc.pl.umap(adata, color=_avail, ncols=4, frameon=False,
                   vmin=0, vmax="p99", show=True,
                   save="_04_marker_genes.png")
        
        # 移动图片
        _src = "figures/umap_04_marker_genes.png"
        _dst = "results/figures/04_marker_genes_umap.png"
        if os.path.exists(_src):
            import shutil
            os.makedirs("results/figures", exist_ok=True)
            shutil.move(_src, _dst)
            print(f"  保存: {_dst}")
        
        print("  解读: 各 marker 应在 UMAP 上形成清晰分离区域")
        print("  如 EPCAM+/PTPRC+ 混在一起 -> 嵌入过校正，考虑降低 Harmony theta 或换 scVI")
    
    # 恢复原有 UMAP
    if _backup_umap is not None:
        adata.obsm["X_umap"] = _backup_umap
    else:
        del adata.obsm["X_umap"]
else:
    print("无可用 UMAP 嵌入，跳过 marker UMAP 叠印")


## 整合指标对比（佐证）

上方已对每个嵌入单独出 UMAP 图供目视判断。本 cell 用定量指标做**补充佐证**——
**UMAP 目测为主决策，指标辅助**。

直接遍历所有已有嵌入，对每个嵌入：
1. 拷贝 adata 避免相互干扰
2. 计算邻居图 + UMAP
3. 调用 `integration_metrics(adata_copy, batch_key=BATCH_KEY, embed_key=rep)` 获取指标
4. 收集结果到对比表

**没有回调、没有 `sweep()`**——学生打开 notebook 能逐行看懂每一步。

指标（数据允许时计算）：
- **silhouette_batch**——越低 = 批次混合越好
- **silhouette_celltype**——越高 = 生物学信号保留越好
- **scib_available**——1.0 表示 scib-metrics 已安装，0.0 表示未安装

对比表写入 `results/figures/sweep_04/sweep_report.md`。
PI 对照 UMAP 图和指标表决定选用哪个嵌入。

In [ ]:
# 显式 for 循环：遍历各嵌入，直接计算 neighbors + UMAP + 整合指标。
# 每一步都是标准 scanpy 操作——学生可以逐行阅读和理解，无需学习回调模型。
print("\n===== 显式遍历嵌入 + 整合指标 =====\n")

import pandas as pd

# 只遍历已计算且存在于 obsm 中的嵌入
use_reps = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI", "X_scCRAFT"]
            if k in adata.obsm]
if not use_reps:
    print("警告: 未在 obsm 中找到任何嵌入。跳过。")
else:
    print(f"遍历 {len(use_reps)} 个嵌入: {use_reps}\n")

    # 基准 k 值（用于自适应缩放）
    _base_k = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]

    results = []
    for rep in use_reps:
        print(f"--- {rep} ---")

        # 拷贝 AnnData 避免不同嵌入之间干扰（邻居图、UMAP 坐标不共用）
        adata_copy = adata.copy()

        # 维度自适应邻居数：低维嵌入用较少邻居，高维嵌入适当增加
        embed_dim = adata_copy.obsm[rep].shape[1]
        _adaptive_k = max(10, int(_base_k * min(1.0, embed_dim / 30)))

        # PCA/Harmony 嵌入空间用 N_PCS_USE 截断；
        # scVI/scANVI/scCRAFT 潜空间本身低维，全部使用
        if rep.startswith("X_sc"):
            n_dim = embed_dim  # 低维潜嵌入用全部维度
        else:
            n_dim = min(N_PCS_USE, embed_dim)

        # 在嵌入空间计算邻居图 + UMAP
        sc.pp.neighbors(adata_copy, use_rep=rep, n_neighbors=_adaptive_k,
                        n_pcs=n_dim, random_state=RANDOM_SEED)
        sc.tl.umap(adata_copy, random_state=RANDOM_SEED)

        # 直接调用整合指标函数——显式传入 embed_key=rep 避免 auto-detect 误选其他嵌入
        m = integration_metrics(adata_copy, batch_key=BATCH_KEY, embed_key=rep)
        # 兼容不同版本 integration_metrics 的返回格式：tuple (name, result_dict) 或纯 dict
        if isinstance(m, tuple):
            # 旧版返回 (embedding_name, metrics_dict) 元组
            _, m = m
        results.append({"use_rep": rep, **m})

        # 输出当前嵌入的指标摘要
        metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items()
                                if isinstance(v, float) and not np.isnan(v))
        print(f"  指标: {metrics_str}")

    # 收集为 DataFrame 对比表
    sweep_df = pd.DataFrame(results)
    os.makedirs("results/figures/sweep_04", exist_ok=True)

    # 写 Markdown 报告（无依赖，纯手写表格）
    lines = ["# 04 嵌入对比报告\n",
             f"**{len(use_reps)} 个嵌入** 已评估。\n",
             "## 指标表\n"]
    lines.append("| " + " | ".join(sweep_df.columns) + " |")
    lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
    for _, row in sweep_df.iterrows():
        vals = []
        for col in sweep_df.columns:
            v = row[col]
            if isinstance(v, float):
                vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    with open("results/figures/sweep_04/sweep_report.md", "w") as f:
        f.write("\n".join(lines) + "\n")

    # 展示对比表
    print("\n整合指标对比表:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(sweep_df)
    except ImportError:
        print(sweep_df)

    print("\n对比报告: results/figures/sweep_04/sweep_report.md")
    adata.uns["04_sweep_v1"] = {
        "embeddings_swept": use_reps,
        "scorer": "integration_metrics",
        "report_dir": "results/figures/sweep_04",
        "timestamp": datetime.datetime.now().isoformat(),
    }

In [ ]:
# --- 过校正检测：检查 batch 校正是否误删了真实生物学差异 ---
# 原理：好的整合 = 同类型跨 batch 混合（好）+ 不同类型仍然分开（保留）
_ct_col = None
for c in ["cell_type_original_Nowicki_2023_v1", "Celltypes_global", "cell_type", "cell_type_final_v1"]:
    if c in adata.obs.columns and adata.obs[c].notna().sum() > 100:
        _ct_col = c
        break

if _ct_col:
    from sklearn.neighbors import NearestNeighbors

    print(f"===== 过校正检测（使用 {_ct_col}）=====")
    # 用最终选定的嵌入（优先 X_pca_harmony，其次 X_pca）
    _embed = adata.obsm.get("X_pca_harmony", adata.obsm.get("X_pca"))
    nn = NearestNeighbors(n_neighbors=50, metric="cosine").fit(_embed)
    _, indices = nn.kneighbors(_embed)

    labels = adata.obs[_ct_col].values
    batches = adata.obs[BATCH_KEY].values

    # 每个细胞的 50 近邻中：同类型不同 batch 的比例（越高越好）
    batch_mix_scores = []
    # 每个细胞的 50 近邻中：不同类型的比例（越低越好）
    type_mix_scores = []

    for i in range(len(labels)):
        neighbors = indices[i, 1:]  # 排除自身
        same_type = labels[neighbors] == labels[i]
        diff_batch = batches[neighbors] != batches[i]
        batch_mix_scores.append((same_type & diff_batch).mean())
        type_mix_scores.append((~same_type).mean())

    batch_mixing = np.mean(batch_mix_scores)
    type_mixing = np.mean(type_mix_scores)
    print(f"  Batch mixing within cell type: {batch_mixing:.3f}（越高越好，说明同类型跨 batch 混合良好）")
    print(f"  Type mixing across neighbors: {type_mixing:.3f}（越低越好，说明不同类型仍然分离）")
    if type_mixing > 0.3:
        print(f"  ⚠️ type_mixing > 0.3：可能存在过校正，不同细胞类型被错误混合")
    elif batch_mixing < 0.1:
        print(f"  ⚠️ batch_mixing < 0.1：batch 效应可能未被充分校正")
    else:
        print(f"  ✓ 整合质量良好：batch 混合充分且类型分离保持")
else:
    print("跳过过校正检测：无可用的 cell_type 列")

In [ ]:
# === 嵌入方法选择建议 ===
# 以下为基于整合指标的自动化建议。**指标是佐证，不是判据**——
# 最终决策仍由 PI 基于 UMAP 目测做出。指标可能被异常细胞/标签
# 噪声干扰，不能替代人对嵌入空间生物学合理性的判断。
print("===== 嵌入方法选择建议 =====")
_available_reps = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI", "X_scCRAFT"]
                   if k in adata.obsm]
print(f"可用嵌入: {_available_reps}")
print()

if 'results' in dir() and len(results) > 1:
    _df = pd.DataFrame(results)
    if "silhouette_batch" in _df.columns and "silhouette_celltype" in _df.columns:
        _best_batch = _df.loc[_df["silhouette_batch"].idxmin(), "use_rep"]
        _best_bio = _df.loc[_df["silhouette_celltype"].idxmax(), "use_rep"]
        print(f"  批次混合最佳: {_best_batch} (silhouette_batch 最低)")
        print(f"  生物学保留最佳: {_best_bio} (silhouette_celltype 最高)")
        if _best_batch == _best_bio:
            print(f"  → 推荐: {_best_batch}（批次混合 + 生物学保留兼优）")
        else:
            print(f"  → 需 PI 目测 UMAP 权衡:")
            print(f"    {_best_batch}（混合好但可能过校正）")
            print(f"    {_best_bio}（分离清晰但可能欠校正）")
        print()
        # 输出可直接复制到 stage 05 的推荐参数
        _recommend = _best_bio if _best_bio != "X_pca" else _best_batch
        print(f"stage 05 推荐设置:")
        print(f'  USE_REP = "{_recommend}"')
    else:
        print("整合指标缺少 silhouette 列——无法生成建议。请目测 UMAP 决定。")
elif len(_available_reps) == 1:
    print(f"仅一种嵌入可用，默认使用: {_available_reps[0]}")
else:
    print("无整合指标结果（可能只跑了 PCA 且无细胞类型标签）")
    print("→ 请目测 UMAP 决定")

## 运行元数据——plain `adata.uns` 写入

版本化键（`harmony_v1`、`scvi_v1` 等）记录每个方法以什么参数运行。
PI 用于追踪溯源和重新运行。

In [ ]:
# 记录每个方法的运行元数据（版本化键，支持重跑时共存）。
print("\n===== 运行元数据 =====")

if "X_pca" in adata.obsm:
    adata.uns["pca_v1"] = {
        "method": "pca",
        "n_comps_computed": N_PCS,      # 总共计算了多少 PC（用于 elbow 诊断）
        "n_pcs_used": N_PCS_USE,        # 实际送入下游的维度数
        "use_hvg": True,
        "svd_solver": "arpack",
        "obsm_key": "X_pca",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_pca_harmony" in adata.obsm:
    adata.uns["harmony_v1"] = {
        "method": "harmony",
        "batch_key": BATCH_KEY,
        "n_pcs_input": N_PCS_USE,       # 喂给 Harmony 的 PCA 维度数
        "n_pcs_output": adata.obsm["X_pca_harmony"].shape[1],  # Harmony 输出维度数
        "obsm_key": "X_pca_harmony",
        "converged": _converged if "_converged" in dir() else True,
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scVI" in adata.obsm:
    adata.uns["scvi_v1"] = {
        "method": "scVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "n_layers": SCVI_N_LAYERS,
        "max_epochs": SCVI_MAX_EPOCHS,
        "gene_likelihood": "zinb",
        "counts_source": _counts_source,  # 实际使用的 counts 来源
        "obsm_key": "X_scVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scANVI" in adata.obsm:
    adata.uns["scanvi_v1"] = {
        "method": "scANVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "counts_source": _counts_source,  # 实际使用的 counts 来源
        "obsm_key": "X_scANVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scCRAFT" in adata.obsm:
    adata.uns["sccraft_v1"] = {
        "method": "scCRAFT",
        "batch_key": BATCH_KEY,
        "epochs": SCCRAFT_EPOCHS,
        "d_coef": SCCRAFT_D_COEF,
        "kl_coef": SCCRAFT_KL_COEF,
        "n_top_genes": SCCRAFT_N_TOP_GENES,
        "cluster_method": SCCRAFT_CLUSTER_METHOD,
        "counts_source": _counts_source,
        "obsm_key": "X_scCRAFT",
        "timestamp": datetime.datetime.now().isoformat(),
    }

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "04_embedded"     # 本 stage 标识
adata.uns["version"] = "v1"            # 与 OUTPUT_PATH 版本号一致

# 上游溯源信息
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# 全局 counts 来源记录——供后续 stage 审计
adata.uns["counts_source"] = _counts_source if "_counts_source" in dir() and _counts_source else "unknown"

# 已运行的方法汇总。
active_embeddings = [k for k in adata.obsm.keys()
                     if k.startswith("X_pca") or k.startswith("X_sc")]
print(f"已产出的嵌入: {active_embeddings}")
print(f"status: {adata.uns['status']}")
for k in sorted(adata.uns.keys()):
    if k.endswith("_v1"):
        print(f"  {k}: {list(adata.uns[k].keys())}")

## Float32 转换

scVI/scANVI 输出默认 float64。转为 float32 内存减半，
对单细胞数据的有效精度无实质影响。

**为什么 float32 够用？** 单细胞 counts 和嵌入携带的信息精度
远低于 float64 的 15 位有效数字。float64 只是白白浪费内存。

In [ ]:
# 将所有 obsm 潜变量矩阵转为 float32。
print("\n===== Float32 转换 =====")
for key in list(adata.obsm.keys()):
    if adata.obsm[key].dtype != np.float32:
        adata.obsm[key] = adata.obsm[key].astype(np.float32)
        print(f"  obsm['{key}'] 转为 float32")
print("全部 obsm dtype:")
for key in adata.obsm:
    print(f"  obsm['{key}']: shape={adata.obsm[key].shape}, dtype={adata.obsm[key].dtype}")

In [ ]:
# 内存自检——写入前一次断言。
# 守卫最高影响的内存退化：adata.X 变 dense 或丢失 float32。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量违反: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32。")

In [ ]:
# 将本 stage 产出写出到磁盘（lzf 压缩——比 gzip 快，比未压缩省约 30% 空间）。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写入 {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

### Stage 04 Verdict

本 stage 完成后应确认：
- [ ] 选定嵌入方法（USE_REP = "..."）-> 复制到 05
- [ ] Marker UMAP 各 compartment 清晰分离
- [ ] 无严重应激 PC 主导


In [ ]:
# 跨 stage 边界释放内存。
# 不释放的话 Jupyter kernel 会一直持有上一 stage 的 AnnData，
# 后续 stage 在同一 kernel 中累积导致 OOM。
del adata
import gc
gc.collect()
print("内存已释放。")